# TDGL Timing Simulation

Set parameters in the cell below, then run all cells to see the timing schedule plot.

In [ ]:
from tdgl_workflow.timing import build_timing_segmented

In [ ]:
# --- Base parameters (shared by all segments) ---
je_initial = 0.0
je_final = 1.0
je_step = 0.1
ramp_time = 100.0
stable_time = 100.0
save_time = 50.0

segments = [
    {"je_initial": je_initial, "je_final": je_final, "je_step": je_step},
    {"je_initial": je_final, "je_final": je_initial, "je_step": -je_step},
]

In [ ]:
# --- Generate timing ---
result = build_timing_segmented(
    segments=segments,
    ramp_time=ramp_time,
    stable_time=stable_time,
    save_time=save_time,
)

steps = result["steps"]
eq = steps[0]
print(f"Total steps: {result['n_steps']} (1 eq + {result['n_steps'] - 1} ramp)")
print(f"Solve time:  {result['solve_time']:.1f}s")
print()
print(f"  eq:  hold je={eq['je_start']:.2f}  t=0.0..{eq['stable_end']:.1f}s")
for i, s in enumerate(steps[1:], 1):
    print(f"  {i:2d}: je {s['je_start']:.2f} -> {s['je_end']:.2f}  "
          f"t={s['ramp_start']:.1f}..{s['stable_end']:.1f}s")

In [ ]:
# --- Plot ---
import plotly.graph_objects as go

times = []
je_vals = []
for s in steps:
    times += [s["ramp_start"], s["ramp_end"], s["stable_end"]]
    je_vals += [s["je_start"], s["je_end"], s["je_end"]]

je_max = max(s["je_end"] for s in steps) or 1.0
band = je_max * 0.06

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=times, y=je_vals, mode="lines",
    line=dict(width=2, color="#2563eb"),
    name="Applied je",
))

# Save windows as fixed-height band centered on je level
for s in steps:
    y_center = s["je_end"] if s["je_end"] != 0 else band
    fig.add_shape(
        type="rect",
        x0=s["save_start"], x1=s["save_end"],
        y0=y_center - band, y1=y_center + band,
        fillcolor="rgba(22,163,74,0.18)", line_width=0,
    )

for s in steps:
    fig.add_vline(x=s["ramp_end"], line_width=0.5,
                  line_dash="dot", line_color="#94a3b8")

fig.update_layout(
    title=f"Timing ({len(steps)} steps, {result['solve_time']:.1f}s)",
    xaxis_title="Time (s)",
    yaxis_title="je",
    legend=dict(orientation="h", yanchor="bottom", y=1.02,
                xanchor="right", x=1),
    margin=dict(l=60, r=20, t=45, b=50),
    height=400, width=800, plot_bgcolor="white",
    xaxis=dict(showline=True, linewidth=1, linecolor="black",
               mirror=True, ticks="outside"),
    yaxis=dict(showline=True, linewidth=1, linecolor="black",
               mirror=True, ticks="outside"),
)
fig.show()